In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given a weighted directed graph of <code>N</code> vertices represented as an
  <code>N</code> &times; <code>N</code> distance matrix, compute the shortest path distance between
  every pair of vertices using the Floyd-Warshall algorithm. The matrix is stored as a flat array in
  row-major order: <code>dist[i * N + j]</code> is the weight of the directed edge from vertex
  <code>i</code> to vertex <code>j</code>. A value of <code>+infinity</code> means no direct edge
  exists. The diagonal is always zero. For each intermediate vertex <code>k</code> from <code>0</code> to <code>N - 1</code>
  (in order), update all pairs:
</p>
<p>
  $$
    \text{output}[i][j] = \min\!\bigl(\text{output}[i][j],\;
      \text{output}[i][k] + \text{output}[k][j]\bigr)
    \quad \forall\, i, j
  $$
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
</ul>

<h2>Example:</h2>
<pre>
Input: N = 4
dist = [
  0,   5, inf,  10,   // row 0: edges from vertex 0
  inf, 0,   3, inf,   // row 1: edges from vertex 1
  inf, inf, 0,   1,   // row 2: edges from vertex 2
  inf, inf, inf, 0    // row 3: edges from vertex 3
]

Output:
output = [
  0,   5,   8,   9,   // shortest paths from vertex 0
  inf, 0,   3,   4,   // shortest paths from vertex 1
  inf, inf, 0,   1,   // shortest paths from vertex 2
  inf, inf, inf, 0    // shortest paths from vertex 3
]

Explanation:
- output[0][2] = 8   (path 0 -&gt; 1 -&gt; 2, cost 5 + 3 = 8)
- output[0][3] = 9   (path 0 -&gt; 1 -&gt; 2 -&gt; 3, cost 5 + 3 + 1 = 9, beats direct 0 -&gt; 3 = 10)
- output[1][3] = 4   (path 1 -&gt; 2 -&gt; 3, cost 3 + 1 = 4)
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 4,096</li>
  <li>Edge weights are finite <code>float32</code> values or <code>+infinity</code> (no edge)</li>
  <li>The input contains no negative cycles</li>
  <li>The diagonal satisfies <code>dist[i * N + i] = 0</code> for all <code>i</code></li>
  <li><code>dist</code> and <code>output</code> are flat arrays of <code>N &times; N</code> floats in row-major order</li>
  <li>Performance is measured with <code>N</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// dist, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* dist, float* output, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# dist, output are tensors on the GPU
@cute.jit
def solve(dist: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# dist is a tensor on the GPU
@jax.jit
def solve(dist: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# dist, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    dist: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# dist, output are tensors on the GPU
def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# dist, output are tensors on the GPU
def solve(dist: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


def _make_graph(N: int, density: float = 0.5, max_weight: float = 10.0, seed: int = None):
    """Create a random non-negative weighted directed graph as a flat float32 CUDA tensor."""
    if seed is not None:
        torch.manual_seed(seed)
    d = torch.full((N * N,), float("inf"), device="cuda", dtype=torch.float32)
    d_view = d.view(N, N)
    d_view.fill_diagonal_(0.0)
    if N > 1:
        mask = torch.rand(N, N, device="cuda") < density
        mask.fill_diagonal_(False)
        weights = torch.rand(N, N, device="cuda") * max_weight + 0.1
        d_view[mask] = weights[mask]
    return d


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="All-Pairs Shortest Paths",
            atol=1e-02,
            rtol=1e-02,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(self, dist: torch.Tensor, output: torch.Tensor, N: int):
        assert dist.shape == (N * N,)
        assert output.shape == (N * N,)
        assert dist.dtype == output.dtype == torch.float32
        assert dist.device == output.device
        assert dist.device.type == "cuda"
        d = dist.view(N, N).clone()
        for k in range(N):
            d = torch.minimum(d, d[:, k : k + 1] + d[k : k + 1, :])
        output.copy_(d.view(-1))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "dist": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        # 4-node directed graph: 0->1:5, 0->3:10, 1->2:3, 2->3:1
        # Shortest paths: 0->2 = 8 (via 1), 0->3 = 9 (via 1->2->3)
        inf = float("inf")
        dist = torch.tensor(
            [0.0, 5.0, inf, 10.0, inf, 0.0, 3.0, inf, inf, inf, 0.0, 1.0, inf, inf, inf, 0.0],
            device="cuda",
            dtype=torch.float32,
        )
        return {
            "dist": dist,
            "output": torch.empty(16, device="cuda", dtype=torch.float32),
            "N": 4,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []
        inf = float("inf")

        def make_output(N):
            return torch.empty(N * N, device="cuda", dtype=torch.float32)

        # --- Edge cases ---

        # N=1: single vertex
        tests.append(
            {
                "dist": torch.tensor([0.0], device="cuda", dtype=torch.float32),
                "output": make_output(1),
                "N": 1,
            }
        )

        # N=2: disconnected graph (no edges between vertices)
        tests.append(
            {
                "dist": torch.tensor([0.0, inf, inf, 0.0], device="cuda", dtype=torch.float32),
                "output": make_output(2),
                "N": 2,
            }
        )

        # N=2: bidirectional edges
        tests.append(
            {
                "dist": torch.tensor([0.0, 3.0, 7.0, 0.0], device="cuda", dtype=torch.float32),
                "output": make_output(2),
                "N": 2,
            }
        )

        # N=3: chain 0->1->2; shortest path 0->2 = 2+3 = 5
        tests.append(
            {
                "dist": torch.tensor(
                    [0.0, 2.0, inf, inf, 0.0, 3.0, inf, inf, 0.0],
                    device="cuda",
                    dtype=torch.float32,
                ),
                "output": make_output(3),
                "N": 3,
            }
        )

        # N=4: graph with shortcut (same as example test)
        tests.append(
            {
                "dist": torch.tensor(
                    [
                        0.0,
                        5.0,
                        inf,
                        10.0,
                        inf,
                        0.0,
                        3.0,
                        inf,
                        inf,
                        inf,
                        0.0,
                        1.0,
                        inf,
                        inf,
                        inf,
                        0.0,
                    ],
                    device="cuda",
                    dtype=torch.float32,
                ),
                "output": make_output(4),
                "N": 4,
            }
        )

        # N=4: negative edge weights, no negative cycles (DAG: 0->1->2->3)
        # 0->1: -1, 1->2: 2, 2->3: -3, 0->3: 10
        # Shortest 0->2 = 1, 0->3 = -2, 1->3 = -1
        tests.append(
            {
                "dist": torch.tensor(
                    [
                        0.0,
                        -1.0,
                        inf,
                        10.0,
                        inf,
                        0.0,
                        2.0,
                        inf,
                        inf,
                        inf,
                        0.0,
                        -3.0,
                        inf,
                        inf,
                        inf,
                        0.0,
                    ],
                    device="cuda",
                    dtype=torch.float32,
                ),
                "output": make_output(4),
                "N": 4,
            }
        )

        # --- Power-of-2 sizes ---
        for N, seed in [(16, 1), (32, 2), (64, 3), (128, 4)]:
            tests.append(
                {
                    "dist": _make_graph(N, density=0.5, seed=seed),
                    "output": make_output(N),
                    "N": N,
                }
            )

        # --- Non-power-of-2 sizes ---
        for N, seed in [(30, 5), (100, 6), (255, 7)]:
            tests.append(
                {
                    "dist": _make_graph(N, density=0.4, seed=seed),
                    "output": make_output(N),
                    "N": N,
                }
            )

        # --- Realistic sizes ---
        for N, seed in [(512, 8)]:
            tests.append(
                {
                    "dist": _make_graph(N, density=0.3, seed=seed),
                    "output": make_output(N),
                    "N": N,
                }
            )

        # --- Special: all zero-weight edges (any path has cost 0) ---
        N = 8
        tests.append(
            {
                "dist": torch.zeros(N * N, device="cuda", dtype=torch.float32),
                "output": make_output(N),
                "N": N,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        N = 2048
        return {
            "dist": _make_graph(N, density=0.3, seed=42),
            "output": torch.empty(N * N, device="cuda", dtype=torch.float32),
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
